# 1. Data Audit


This section performs an initial audit of the raw HR Attrition dataset. It covers: dataset shape, column names and data types, missing value counts, duplicate row detection, unique category values for categorical and low-cardinality numeric columns, a numeric summary (min/max/mean/std), the target class distribution (Attrition Yes/No), and a full `df.info()` printout.


In [ ]:
import sys
import pandas as pd
import numpy as np
sys.stdout.reconfigure(encoding="utf-8")

df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

DIVIDER = "=" * 60
SEP     = "-" * 60

# Shape
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

# Column names
print("\nColumns:", df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes.to_string())

# Missing values
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0]
print("\nMissing Values:")
print("None found." if missing_df.empty else missing_df.to_string())
print(f"\nTotal missing: {df.isna().sum().sum():,}")

# Duplicates
print(f"\nDuplicate rows: {df.duplicated().sum():,}")

# Unique categories
cat_cols = df.select_dtypes(include=["str", "object", "category"]).columns.tolist()
num_quasi = [c for c in df.select_dtypes(include="number").columns if df[c].nunique() <= 10]
print("\nUnique Categories:")
for col in cat_cols + num_quasi:
    vals = sorted(str(v) for v in df[col].dropna().unique())
    display = ", ".join(vals[:10]) + (f"  ...+{len(vals)-10}" if len(vals) > 10 else "")
    print(f"  {col:<30} unique={df[col].nunique():>4}   [{display}]")

# Numeric summary
print("\nNumeric Summary:")
num_cols = df.select_dtypes(include="number").columns.tolist()
print(df[num_cols].agg(["min","max","mean","std"]).T.round(2).to_string())

# Class distribution
print("\nClass Distribution - Attrition:")
counts = df["Attrition"].value_counts(dropna=False)
pcts   = df["Attrition"].value_counts(normalize=True, dropna=False).mul(100).round(2)
print(pd.DataFrame({"Count": counts, "Percent %": pcts}).to_string())

# df.info()
print("\ndf.info():")
df.info()


# 2. Data Cleaning


This section cleans the raw dataset. Steps include: missing value imputation (median for numeric, mode for categorical), duplicate row removal, invalid value checks (age range 18–65, negative income/rates, impossible tenure relationships), valid category enforcement, type conversions (binary booleans, category dtypes), and dropping zero-variance columns (`EmployeeCount`, `Over18`, `StandardHours`). The cleaned dataset is saved as `hr_attrition_clean.csv`.


In [ ]:
import sys
import pandas as pd
import numpy as np
sys.stdout.reconfigure(encoding="utf-8")

df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.columns = df.columns.str.strip().str.lstrip("\ufeff")

# Missing values
missing = df.isna().sum()
missing = missing[missing > 0]
if not missing.empty:
    for col in missing.index:
        if df[col].dtype == "object":
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())
print(f"Missing after clean: {df.isna().sum().sum()}")

# Duplicates
n_before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {n_before - len(df)}")

# Invalid values
checks = {
    "Age": lambda d: d["Age"].between(18, 65),
    "MonthlyIncome": lambda d: d["MonthlyIncome"] > 0,
    "DailyRate": lambda d: d["DailyRate"] > 0,
    "HourlyRate": lambda d: d["HourlyRate"] > 0,
    "MonthlyRate": lambda d: d["MonthlyRate"] > 0,
    "Tenure logic": lambda d: d["YearsAtCompany"] <= d["TotalWorkingYears"],
    "Role tenure": lambda d: d["YearsInCurrentRole"] <= d["YearsAtCompany"],
    "Manager tenure": lambda d: d["YearsWithCurrManager"] <= d["YearsAtCompany"],
}
for name, fn in checks.items():
    mask = fn(df)
    dropped = (~mask).sum()
    df = df[mask]
    print(f"  {name:<25}: {dropped} rows dropped" if dropped else f"  {name:<25}: OK")

# Valid categories
valid_cats = {
    "Attrition": {"Yes","No"}, "BusinessTravel": {"Non-Travel","Travel_Rarely","Travel_Frequently"},
    "Department": {"Sales","Research & Development","Human Resources"},
    "Gender": {"Male","Female"}, "MaritalStatus": {"Single","Married","Divorced"},
    "OverTime": {"Yes","No"}, "Over18": {"Y"},
    "EducationField": {"Life Sciences","Medical","Marketing","Technical Degree","Human Resources","Other"},
    "JobRole": {"Sales Executive","Research Scientist","Laboratory Technician","Manufacturing Director",
                "Healthcare Representative","Manager","Sales Representative","Research Director","Human Resources"},
}
for col, valid in valid_cats.items():
    mask = df[col].isin(valid)
    dropped = (~mask).sum()
    df = df[mask]
    print(f"  {col:<20}: {dropped} invalid rows dropped" if dropped else f"  {col:<20}: OK")

# Type conversions
binary_map = {"Yes": True, "No": False}
df["Attrition"] = df["Attrition"].map(binary_map)
df["OverTime"]  = df["OverTime"].map(binary_map)
for col in ["BusinessTravel","Department","EducationField","Gender","JobRole","MaritalStatus"]:
    df[col] = df[col].astype("category")

# Drop zero-variance columns
df = df.drop(columns=["EmployeeCount", "Over18", "StandardHours"])

print(f"\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Missing: {df.isna().sum().sum()}  |  Duplicates: {df.duplicated().sum()}")
df.to_csv("hr_attrition_clean.csv", index=False)
print("Saved: hr_attrition_clean.csv")


# 3. Exploratory Data Analysis


This section explores attrition patterns across demographics (age, gender, marital status, education), job attributes (department, role, level, business travel), compensation (income, salary band, stock options), workplace factors (overtime, job satisfaction, environment satisfaction, work-life balance), and career trajectory (tenure band, years with manager). Point-biserial correlations with the Attrition target are also computed for all numeric features.


In [ ]:
import sys
import pandas as pd
import numpy as np
sys.stdout.reconfigure(encoding="utf-8")

df = pd.read_csv("hr_attrition_clean.csv")
df["Attrition"] = df["Attrition"].astype(str).map({"True": True, "False": False})
df["Attrition"] = df["Attrition"].astype(bool)

def attrition_table(df, col):
    grp   = df.groupby(col, observed=True)
    total = grp["Attrition"].count().rename("Total")
    left  = grp["Attrition"].sum().rename("Left")
    rate  = (left / total * 100).round(1).rename("Attrition %")
    return pd.concat([total, left, rate], axis=1).sort_values("Attrition %", ascending=False)

# Overall attrition
total_employees = len(df)
employees_left  = df["Attrition"].sum()
print(f"Total: {total_employees:,}  |  Left: {int(employees_left):,}  |  "
      f"Rate: {employees_left/total_employees*100:.2f}%")

# Demographics
bins   = [17, 24, 34, 44, 54, 65]
labels = ["18-24","25-34","35-44","45-54","55-65"]
df["AgeBand"] = pd.cut(df["Age"], bins=bins, labels=labels)
print("\nAttrition by Age Band:")
print(attrition_table(df, "AgeBand").to_string())

print("\nAttrition by Gender:")
print(attrition_table(df, "Gender").to_string())

print("\nAttrition by Marital Status:")
print(attrition_table(df, "MaritalStatus").to_string())

edu_map = {1:"1-Below College",2:"2-College",3:"3-Bachelor",4:"4-Master",5:"5-Doctor"}
df["EducationLabel"] = df["Education"].astype(int).map(edu_map)
print("\nAttrition by Education Level:")
print(attrition_table(df, "EducationLabel").to_string())

# Job
print("\nAttrition by Department:")
print(attrition_table(df, "Department").to_string())

print("\nAttrition by Job Role:")
print(attrition_table(df, "JobRole").to_string())

lvl_map = {1:"1-Entry",2:"2-Junior",3:"3-Mid",4:"4-Senior",5:"5-Director"}
df["JobLevelLabel"] = df["JobLevel"].astype(int).map(lvl_map)
print("\nAttrition by Job Level:")
print(attrition_table(df, "JobLevelLabel").to_string())

print("\nAttrition by Business Travel:")
print(attrition_table(df, "BusinessTravel").to_string())

# Compensation
print("\nMonthly Income (Left vs Stayed):")
inc = df.groupby("Attrition", observed=True)["MonthlyIncome"].agg(["mean","median"]).round(0)
inc.index = ["Stayed","Left"]
print(inc.to_string())

inc_bins   = [0,3000,6000,9000,12000,21000]
inc_labels = ["<3k","3k-6k","6k-9k","9k-12k",">12k"]
df["SalaryBand"] = pd.cut(df["MonthlyIncome"], bins=inc_bins, labels=inc_labels)
print("\nAttrition by Salary Band:")
print(attrition_table(df, "SalaryBand").to_string())

sol_map = {0:"0-None",1:"1-Low",2:"2-Medium",3:"3-High"}
df["StockLabel"] = df["StockOptionLevel"].astype(int).map(sol_map)
print("\nAttrition by Stock Options:")
print(attrition_table(df, "StockLabel").to_string())

# Workplace
ot_map = {True:"Yes", False:"No"}
df["OvertimeLabel"] = df["OverTime"].map(ot_map)
print("\nAttrition by Overtime:")
print(attrition_table(df, "OvertimeLabel").to_string())

sat_map = {1:"1-Low",2:"2-Medium",3:"3-High",4:"4-Very High"}
df["JobSatLabel"] = df["JobSatisfaction"].astype(int).map(sat_map)
print("\nAttrition by Job Satisfaction:")
print(attrition_table(df, "JobSatLabel").to_string())

df["EnvSatLabel"] = df["EnvironmentSatisfaction"].astype(int).map(sat_map)
print("\nAttrition by Environment Satisfaction:")
print(attrition_table(df, "EnvSatLabel").to_string())

wlb_map = {1:"1-Bad",2:"2-Good",3:"3-Better",4:"4-Best"}
df["WLBLabel"] = df["WorkLifeBalance"].astype(int).map(wlb_map)
print("\nAttrition by Work-Life Balance:")
print(attrition_table(df, "WLBLabel").to_string())

# Career
yac_bins   = [-1,1,3,5,10,20,41]
yac_labels = ["0-1","2-3","4-5","6-10","11-20","21+"]
df["TenureBand"] = pd.cut(df["YearsAtCompany"], bins=yac_bins, labels=yac_labels)
print("\nAttrition by Tenure Band:")
print(attrition_table(df, "TenureBand").to_string())

mgr_bins   = [-1,0,2,5,10,18]
mgr_labels = ["0","1-2","3-5","6-10","11+"]
df["MgrBand"] = pd.cut(df["YearsWithCurrManager"], bins=mgr_bins, labels=mgr_labels)
print("\nAttrition by Years with Manager:")
print(attrition_table(df, "MgrBand").to_string())

# Correlations
numeric_features = ["Age","DailyRate","DistanceFromHome","HourlyRate","MonthlyIncome",
    "MonthlyRate","NumCompaniesWorked","PercentSalaryHike","TotalWorkingYears",
    "TrainingTimesLastYear","YearsAtCompany","YearsInCurrentRole",
    "YearsSinceLastPromotion","YearsWithCurrManager"]
corr_df = df[numeric_features + ["Attrition"]].copy()
corr_df["Attrition"] = corr_df["Attrition"].astype(int)
print("\nPoint-biserial correlations with Attrition:")
print(corr_df.corr()["Attrition"].drop("Attrition").sort_values().round(4).to_string())


# 4. Feature Engineering


This section creates four new engineered features from existing columns: `SalaryBand` (tertile-based income bands: Low/Medium/High), `AgeGroup` (binned age ranges), `TenureGroup` (binned years at company), and `PromotionGap` (recent vs long gap since last promotion). Each feature is profiled with attrition counts and rates. The enriched dataset is saved as `hr_attrition_features.csv`.


In [ ]:
import sys
import pandas as pd
import numpy as np
sys.stdout.reconfigure(encoding="utf-8")

df = pd.read_csv("hr_attrition_clean.csv")
df["Attrition"] = df["Attrition"].astype(str).map({"True": True, "False": False})

# Salary Band (tertile-based)
tertiles = df["MonthlyIncome"].quantile([1/3, 2/3])
low_cap, mid_cap = tertiles[1/3], tertiles[2/3]
df["SalaryBand"] = pd.cut(df["MonthlyIncome"], bins=[-np.inf, low_cap, mid_cap, np.inf],
                          labels=["Low", "Medium", "High"])

# Age Group
df["AgeGroup"] = pd.cut(df["Age"], bins=[17,25,35,45,55,65],
                        labels=["18-25","26-35","36-45","46-55","56+"])

# Tenure Group
df["TenureGroup"] = pd.cut(df["YearsAtCompany"], bins=[-1,1,3,5,10,np.inf],
                           labels=["0-1","2-3","4-5","6-10","10+"])

# Promotion Gap
df["PromotionGap"] = pd.cut(df["YearsSinceLastPromotion"], bins=[-1,2,np.inf],
                            labels=["Recent (0-2 yrs)","Long (3+ yrs)"])

# Profile each feature
new_cols = ["SalaryBand","AgeGroup","TenureGroup","PromotionGap"]
for col in new_cols:
    grp = (df.groupby(col, observed=True)["Attrition"]
           .agg(Total="count", Left="sum")
           .assign(**{"Attrition %": lambda x: (x["Left"]/x["Total"]*100).round(1)})
           .sort_values("Attrition %", ascending=False))
    print(f"\n{col}:")
    print(grp.to_string())

df.to_csv("hr_attrition_features.csv", index=False)
print(f"\nSaved: hr_attrition_features.csv  ({df.shape[0]:,} rows x {df.shape[1]} cols)")


# 5. Hypothesis Testing


Ten statistical hypotheses are tested at significance level α = 0.05. Chi-squared tests (χ²) are used for categorical variables (Overtime, Marital Status, Business Travel, Job Level, Stock Options). Mann-Whitney U tests are used for continuous/ordinal variables (Monthly Income, Age, Job Satisfaction, Years at Company, Distance from Home). A summary table of all results (test statistic, p-value, decision) is printed at the end.


In [ ]:
import sys
import pandas as pd
import numpy as np
from scipy import stats
sys.stdout.reconfigure(encoding="utf-8")

ALPHA = 0.05
df = pd.read_csv("hr_attrition_features.csv")
df["Attrition"] = df["Attrition"].astype(str).map({"True": True, "False": False})
stayed = df[df["Attrition"] == False]
left   = df[df["Attrition"] == True]

def verdict(p):
    return f"REJECT H0 (p={p:.4f})" if p < ALPHA else f"FAIL TO REJECT H0 (p={p:.4f})"

def chi2_test(col):
    ct = pd.crosstab(df[col], df["Attrition"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    return chi2, p, dof

def mwu_test(col):
    u, p = stats.mannwhitneyu(stayed[col].dropna(), left[col].dropna(), alternative="two-sided")
    return u, p

hypotheses = [
    ("H1",  "Overtime",           "chi2",  "OverTime",
     "Overtime and attrition are independent.",
     "Overtime and attrition are associated."),
    ("H2",  "Monthly Income",     "mwu",   "MonthlyIncome",
     "Income distribution is the same for leavers and stayers.",
     "Income distribution differs between groups."),
    ("H3",  "Age",                "mwu",   "Age",
     "Age distribution is the same for leavers and stayers.",
     "Age distribution differs between groups."),
    ("H4",  "Marital Status",     "chi2",  "MaritalStatus",
     "Marital status and attrition are independent.",
     "Marital status and attrition are associated."),
    ("H5",  "Business Travel",    "chi2",  "BusinessTravel",
     "Business travel and attrition are independent.",
     "Business travel and attrition are associated."),
    ("H6",  "Job Level",          "chi2",  "JobLevel",
     "Job level and attrition are independent.",
     "Job level and attrition are associated."),
    ("H7",  "Job Satisfaction",   "mwu",   "JobSatisfaction",
     "Job satisfaction is the same for leavers and stayers.",
     "Job satisfaction differs between groups."),
    ("H8",  "Years at Company",   "mwu",   "YearsAtCompany",
     "Tenure is the same for leavers and stayers.",
     "Tenure differs between groups."),
    ("H9",  "Stock Options",      "chi2",  "StockOptionLevel",
     "Stock option level and attrition are independent.",
     "Stock option level and attrition are associated."),
    ("H10", "Distance from Home", "mwu",   "DistanceFromHome",
     "Distance from home is the same for leavers and stayers.",
     "Distance from home differs between groups."),
]

results = []
for hid, name, test, col, h0, h1 in hypotheses:
    print(f"\n{'='*55}")
    print(f"  {hid}: {name}")
    print(f"  H0: {h0}")
    print(f"  H1: {h1}")
    print(f"  {'─'*50}")
    if test == "chi2":
        stat, p, dof = chi2_test(col)
        print(f"  Test: Chi-squared  |  chi2={stat:.4f}  dof={dof}")
    else:
        stat, p = mwu_test(col)
        grp = df.groupby("Attrition")[col].agg(["mean","median"]).round(2)
        grp.index = ["Stayed","Left"]
        print(f"  Test: Mann-Whitney U  |  U={stat:.0f}")
        print(f"  {grp.to_string()}")
    print(f"  {verdict(p)}")
    results.append({"ID": hid, "Variable": name, "Test": test.upper(), "p-value": round(p, 4), "Decision": "REJECT H0" if p < ALPHA else "FAIL"})

print(f"\n{'='*55}")
print("SUMMARY TABLE")
print(pd.DataFrame(results).to_string(index=False))


# 6. SQL Database & Analysis


This section loads the feature-engineered dataset into a SQLite database and runs 13 SQL queries. Queries cover: total headcount, attrition counts and rates, income stats, attrition by department/role/overtime, CTEs for role-vs-company deviation and high-risk profile analysis, income quartile CASE expressions, a running attrition total window function, top-3 paid leavers per department using RANK() OVER PARTITION BY, and an overtime × marital status cross-tab.


In [ ]:
import sys
import sqlite3
import pandas as pd
sys.stdout.reconfigure(encoding="utf-8")

df = pd.read_csv("hr_attrition_features.csv")
df["Attrition"] = df["Attrition"].astype(str).map({"True": 1, "False": 0})
df["OverTime"]  = df["OverTime"].astype(str).map({"True": 1, "False": 0})
df.columns = df.columns.str.strip().str.lstrip("\ufeff")

conn = sqlite3.connect("hr_attrition.db")
df.drop(columns=["EmployeeNumber","SalaryBand","AgeGroup","TenureGroup","PromotionGap"],
        errors="ignore").to_sql("employees", conn, if_exists="replace", index=False)
print(f"Loaded {len(df):,} rows into 'employees' table")

queries = {
    "Q01 Total employees":
        "SELECT COUNT(*) AS total FROM employees",
    "Q02 Left vs Stayed":
        "SELECT CASE WHEN Attrition=1 THEN 'Left' ELSE 'Stayed' END AS status, COUNT(*) AS count FROM employees GROUP BY Attrition",
    "Q03 Attrition rate %":
        "SELECT ROUND(SUM(Attrition)*100.0/COUNT(*),2) AS attrition_pct FROM employees",
    "Q04 Income stats":
        "SELECT ROUND(AVG(MonthlyIncome),0) AS avg, MIN(MonthlyIncome) AS min, MAX(MonthlyIncome) AS max FROM employees",
    "Q05 Attrition by department":
        "SELECT Department, COUNT(*) AS total, SUM(Attrition) AS left_count, ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS pct FROM employees GROUP BY Department ORDER BY pct DESC",
    "Q06 Attrition by job role":
        "SELECT JobRole, COUNT(*) AS total, SUM(Attrition) AS left_count, ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS pct FROM employees GROUP BY JobRole ORDER BY pct DESC",
    "Q07 Attrition by overtime":
        "SELECT CASE WHEN OverTime=1 THEN 'Yes' ELSE 'No' END AS overtime, COUNT(*) AS total, SUM(Attrition) AS left_count, ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS pct FROM employees GROUP BY OverTime ORDER BY pct DESC",
    "Q08 CTE - role vs company avg":
        "WITH role_stats AS (\n    SELECT JobRole, COUNT(*) AS total, SUM(Attrition) AS left_count,\n           ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS role_pct FROM employees GROUP BY JobRole\n), company_avg AS (\n    SELECT ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS overall_pct FROM employees\n)\nSELECT r.JobRole, r.total, r.role_pct, c.overall_pct,\n       ROUND(r.role_pct - c.overall_pct, 1) AS deviation\nFROM role_stats r CROSS JOIN company_avg c ORDER BY deviation DESC",
    "Q09 High-risk profile CTE":
        "WITH high_risk AS (SELECT * FROM employees WHERE JobLevel=1 AND OverTime=1 AND JobSatisfaction<=2)\nSELECT COUNT(*) AS count, SUM(Attrition) AS left_count,\n       ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS attrition_pct,\n       ROUND(AVG(MonthlyIncome),0) AS avg_income FROM high_risk",
    "Q10 Income quartile CASE":
        "SELECT CASE WHEN MonthlyIncome<=2911 THEN 'Q1 Bottom 25%'\n                   WHEN MonthlyIncome<=4919 THEN 'Q2 Lower-Mid'\n                   WHEN MonthlyIncome<=8379 THEN 'Q3 Upper-Mid'\n                   ELSE 'Q4 Top 25%' END AS quartile,\n           COUNT(*) AS total, SUM(Attrition) AS left_count,\n           ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS pct\nFROM employees GROUP BY quartile ORDER BY quartile",
    "Q11 Running attrition by tenure (window)":
        "SELECT TenureGroup,\n       SUM(Attrition) AS left_in_band,\n       SUM(SUM(Attrition)) OVER (\n           ORDER BY CASE TenureGroup WHEN '0-1' THEN 1 WHEN '2-3' THEN 2\n           WHEN '4-5' THEN 3 WHEN '6-10' THEN 4 WHEN '10+' THEN 5 END\n           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n       ) AS running_total, COUNT(*) AS headcount\nFROM employees\nGROUP BY TenureGroup\nORDER BY CASE TenureGroup WHEN '0-1' THEN 1 WHEN '2-3' THEN 2\n         WHEN '4-5' THEN 3 WHEN '6-10' THEN 4 WHEN '10+' THEN 5 END",
    "Q12 Top 3 paid leavers per dept (window)":
        "SELECT * FROM (\n    SELECT EmployeeNumber, Department, JobRole, MonthlyIncome, YearsAtCompany,\n           RANK() OVER (PARTITION BY Department ORDER BY MonthlyIncome DESC) AS pay_rank\n    FROM employees WHERE Attrition=1\n) WHERE pay_rank<=3 ORDER BY Department, pay_rank",
    "Q13 Overtime x Marital cross-tab":
        "SELECT CASE WHEN OverTime=1 THEN 'Yes' ELSE 'No' END AS overtime, MaritalStatus,\n       COUNT(*) AS total, SUM(Attrition) AS left_count,\n       ROUND(SUM(Attrition)*100.0/COUNT(*),1) AS pct\nFROM employees GROUP BY OverTime, MaritalStatus ORDER BY pct DESC",
}

for title, sql in queries.items():
    print(f"\n--- {title} ---")
    result = pd.read_sql_query(sql, conn)
    print(result.to_string(index=False))

conn.close()


# 7. Build ML Dataset


This section prepares the machine-learning-ready dataset from the feature-engineered CSV. Steps include: dropping non-feature columns (EmployeeNumber and derived bin columns), casting ordinal columns to integers, one-hot encoding nominal categorical columns (drop_first=True), and separating features (X) and target (y). The class imbalance ratio is reported and all feature names are listed. The final ML dataset is saved as `ml_dataset.csv`.


In [ ]:
import sys
import pandas as pd
import numpy as np
sys.stdout.reconfigure(encoding="utf-8")

df = pd.read_csv("hr_attrition_features.csv")
df["Attrition"] = df["Attrition"].astype(str).map({"True": 1, "False": 0})
df["OverTime"]  = df["OverTime"].astype(str).map({"True": 1, "False": 0})

# Drop columns
DROP_COLS = ["EmployeeNumber","SalaryBand","AgeGroup","TenureGroup","PromotionGap"]
df = df.drop(columns=DROP_COLS)

# Ordinal — keep as int
ordinal_cols = ["Education","EnvironmentSatisfaction","JobInvolvement","JobLevel",
                "JobSatisfaction","PerformanceRating","RelationshipSatisfaction",
                "StockOptionLevel","WorkLifeBalance"]
for col in ordinal_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype(int)

# One-hot encode nominal columns
nominal_cols = ["BusinessTravel","Department","EducationField","Gender","JobRole","MaritalStatus"]
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

# Separate X and y
y = df["Attrition"].astype(int)
X = df.drop(columns=["Attrition"])

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:")
counts = y.value_counts().sort_index()
pcts   = y.value_counts(normalize=True).sort_index().mul(100).round(2)
print(pd.DataFrame({"Label":["Stayed (0)","Left (1)"], "Count":counts.values, "Pct %":pcts.values}).to_string(index=False))
print(f"\nImbalance ratio: {counts[0]/counts[1]:.1f}:1")
print(f"\nFeatures ({X.shape[1]}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:>3}. {col}")

ml_df = X.copy()
ml_df.insert(0, "Attrition", y)
ml_df.to_csv("ml_dataset.csv", index=False)
print(f"\nSaved: ml_dataset.csv")


# 8. Prevent Data Leakage — Leakage-Safe Pipeline


This section implements a leakage-safe preprocessing pipeline following best practices. The train/test split is performed BEFORE any preprocessing. A `ColumnTransformer` is then fitted exclusively on the training set: `StandardScaler` is applied to numeric features and `OneHotEncoder` (drop='first', handle_unknown='ignore') to nominal categorical features. The fitted preprocessor is then used only to transform the test set. All processed splits and the fitted preprocessor are persisted to disk. Seven leakage checks are verified and reported.


In [ ]:
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
sys.stdout.reconfigure(encoding="utf-8")

RANDOM_STATE = 42
TEST_SIZE    = 0.20

df = pd.read_csv("hr_attrition_features.csv")
df["Attrition"] = df["Attrition"].astype(str).map({"True": 1, "False": 0})
df["OverTime"]  = df["OverTime"].astype(str).map({"True": 1, "False": 0})

# Drop leaky columns
ALL_DROP = ["EmployeeNumber","SalaryBand","AgeGroup","TenureGroup","PromotionGap"]
df = df.drop(columns=[c for c in ALL_DROP if c in df.columns])

y = df["Attrition"].astype(int)
X = df.drop(columns=["Attrition"])

# *** SPLIT BEFORE PREPROCESSING ***
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"Train attrition: {y_train.mean()*100:.1f}%  |  Test attrition: {y_test.mean()*100:.1f}%")

# Column classification
NOMINAL_COLS = ["BusinessTravel","Department","EducationField","Gender","JobRole","MaritalStatus"]
NUMERIC_COLS = [c for c in X.columns if c not in NOMINAL_COLS]

# Pipeline
numeric_transformer = Pipeline([("scaler", StandardScaler())])
nominal_transformer = Pipeline([("onehot", OneHotEncoder(
    handle_unknown="ignore", drop="first", sparse_output=False))])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, NUMERIC_COLS),
    ("cat", nominal_transformer, NOMINAL_COLS),
], remainder="drop")

# *** FIT ON TRAIN ONLY ***
preprocessor.fit(X_train)
X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f"\nX_train_proc: {X_train_proc.shape}")
print(f"X_test_proc : {X_test_proc.shape}")

# Feature names
ohe_names = (preprocessor.named_transformers_["cat"]
             .named_steps["onehot"].get_feature_names_out(NOMINAL_COLS).tolist())
all_features = NUMERIC_COLS + ohe_names
print(f"\nTotal features after encoding: {len(all_features)}")

# Save artefacts
pd.DataFrame(X_train_proc, columns=all_features).to_csv("X_train.csv", index=False)
pd.DataFrame(X_test_proc,  columns=all_features).to_csv("X_test.csv",  index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv",   index=False)
joblib.dump(preprocessor, "preprocessor.pkl")

print("\nLeakage checks:")
checks = [
    "Preprocessor fitted on X_train only",
    "X_test passed to .transform() only, never .fit()",
    "stratify=y preserves class ratio in both splits",
    "EmployeeNumber (identifier) dropped before split",
    "Derived bins dropped; numeric sources retained",
    "handle_unknown='ignore' handles unseen test categories safely",
    "No post-event columns present in dataset",
]
for c in checks:
    print(f"  [OK] {c}")

print("\nAll artefacts saved.")
